In [ ]:
# Librairies de base
import sqlite3
import pandas as pd
import numpy as np
import os
import easyocr
import difflib  # Pour la recherche floue (Text Matching)

print("Librairies importées avec succès. EasyOCR est prêt.")

Librairies importées avec succès.


In [ ]:
# Configuration des chemins
db_path = os.path.join('..', 'data', 'bdpm.db')

print(f"Configuration des chemins terminée. Base ciblée : {db_path}")

Configuration des chemins terminée.


In [3]:
#  Vérification de l'existence du fichier de base de données

# Get the absolute path to the database file
abs_db_path = os.path.abspath(db_path);

print(f"Current working directory: {os.getcwd()}")
print(f"Looking for database at absolute path: {abs_db_path}")

if not os.path.exists(abs_db_path):
    print(f"ERREUR: Le fichier de base de données n'existe pas à l'emplacement spécifié : {abs_db_path}")
    print("Veuillez vous assurer que le fichier 'bdpm.db' est bien dans le dossier '../data/' ou ajustez le chemin.")
else:
    print(f"Fichier de base de données trouvé à : {abs_db_path}")
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Lister toutes les tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"Tables trouvées : {tables}")

    # Si des tables existent, lister les colonnes de la première table trouvée
    if tables:
        table_name = tables[0][0]
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        print(f"Colonnes de la table '{table_name}' : {[col[1] for col in columns]}")
        print(f"Affichage de quelques lignes de la table '{table_name}' :")
        cursor.execute(f"SELECT * FROM {table_name} LIMIT 5;")
        rows = cursor.fetchall()
        for row in rows:
            print(row)
    conn.close()

Current working directory: c:\Users\mgraz\Documents\.vscode\Dépot GitHub\medication-webapp\ocr
Looking for database at absolute path: c:\Users\mgraz\Documents\.vscode\Dépot GitHub\medication-webapp\data\bdpm.db
Fichier de base de données trouvé à : c:\Users\mgraz\Documents\.vscode\Dépot GitHub\medication-webapp\data\bdpm.db
Tables trouvées : [('medicaments',), ('presentations',), ('compositions',), ('conditions_prescription',), ('generiques',)]
Colonnes de la table 'medicaments' : ['CIS', 'DENOMINATION', 'FORME', 'VOIES', 'STATUT_AMM', 'TYPE_PROC', 'ETAT_COMM', 'DATE_AMM', 'STATUT_BDM', 'NUM_AMM', 'TITULAIRES', 'SURVEILLANCE']
Affichage de quelques lignes de la table 'medicaments' :
('61266250', 'A 313 200 000 UI POUR CENT, POMMADE', 'POMMADE', 'CUTANEE', 'AUTORISATION ACTIVE', 'PROCEDURE NATIONALE', 'COMMERCIALISEE', '12/03/1998', None, None, 'PHARMA DEVELOPPEMENT', 'NON')
('62869109', 'A 313 50 000 U.I., CAPSULE MOLLE', 'CAPSULE MOLLE', 'ORALE', 'AUTORISATION ACTIVE', 'PROCEDURE NATI

In [12]:
# 1. Fonction pour charger les données
def load_data_from_db():
    if not os.path.exists(db_path):
        print(f"ERREUR : Le fichier n'existe pas ici : {os.path.abspath(db_path)}")
        return None

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Directly use the 'medicaments' table, which is known to contain 'CIS' and 'DENOMINATION'
    table_name = 'medicaments'
    print(f"Table détectée : {table_name}")

    # Adapt query to the found table name and expected columns
    query = f"SELECT CIS, DENOMINATION FROM {table_name};"

    try:
        df = pd.read_sql_query(query, conn)
        # Rename 'DENOMINATION' to 'Designation' to match the preprocess_data function's expectation
        df = df.rename(columns={'DENOMINATION': 'Designation'})
    except Exception as e:
        print(f"Erreur lors du chargement des données depuis la table {table_name}: {e}")
        print("Vérifiez que les colonnes 'CIS' et 'DENOMINATION' existent.")
        conn.close()
        return None

    conn.close()
    return df
print("Fonction de chargement des données définie.")

df = load_data_from_db()
if df is not None:
    print(f"Nombre de lignes chargées : {len(df)}")
    print("Aperçu des données :")
    print(df.head())
else:
    print("Aucune donnée chargée. Veuillez vérifier les erreurs précédentes.")

Fonction de chargement des données définie.
Table détectée : medicaments
Nombre de lignes chargées : 15822
Aperçu des données :
        CIS                                        Designation
0  61266250                A 313 200 000 UI POUR CENT, POMMADE
1  62869109                   A 313 50 000 U.I., CAPSULE MOLLE
2  69103878  A.D.N. BOIRON, DEGRE DE DILUTION COMPRIS ENTRE...
3  61876780  ABACAVIR ARROW 300 MG, COMPRIME PELLICULE SECABLE
4  63797011  ABACAVIR SANDOZ 300 MG, COMPRIME PELLICULE SEC...


In [ ]:
def find_best_medicine(ocr_text, dataframe, threshold=0.6):
    """
    Compare le texte issu de l'OCR avec la base de données BDPM
    et retourne le médicament le plus ressemblant.
    """
    # Nettoyage du texte recherché
    query = ocr_text.lower().strip()
    
    # On récupère toutes les dénominations de la BDD en minuscules
    all_denominations = dataframe['Designation'].str.lower().tolist()
    
    # Extraction de la meilleure correspondance floue
    matches = difflib.get_close_matches(query, all_denominations, n=1, cutoff=threshold)
    
    if matches:
        best_match = matches[0]
        # On retrouve la ligne correspondante dans le dataframe pour avoir le CIS
        result_row = dataframe[dataframe['Designation'].str.lower() == best_match].iloc[0]
        
        # Calcul du score de confiance exact (ratio entre 0 et 1)
        score = difflib.SequenceMatcher(None, query, best_match).ratio()
        
        return {
            "CIS": result_row['CIS'],
            "DENOMINATION": result_row['Designation'],
            "Confiance": score
        }
    
    return None

print("Moteur de correspondance textuelle configuré.")

Fonction de prétraitement définie.
Toutes les fonctions sont prêtes. Vous pouvez maintenant charger les données et les prétraiter.
Exécutez : df = load_data_from_db()
Ensuite : X, y, tokenizer, label_tokenizer = preprocess_data(df)


In [ ]:
def process_prescription_image(image_path, dataframe):
    print("Initialisation d'EasyOCR (mode CPU)...")
    # Initialisation pour le Français (fr) et l'Anglais (en)
    reader = easyocr.Reader(['fr', 'en'], gpu=False)
    
    print(f"Analyse de l'image : {image_path}")
    # Extraction du texte brut (detail=0 renvoie juste une liste de chaînes de caractères)
    ocr_results = reader.readtext(image_path, detail=0)
    
    print("\n--- Textes détectés par l'OCR ---")
    print(ocr_results)
    print("---------------------------------\n")
    
    final_results = []
    
    # On parcours chaque ligne ou mot trouvé par l'OCR pour chercher un médicament
    for text_line in ocr_results:
        # On ignore les textes trop courts (ex: "1", "g", "par")
        if len(text_line) < 4:
            continue
            
        match = find_best_medicine(text_line, dataframe, threshold=0.5)
        if match:
            final_results.append(match)
            
    return final_results

# --- Zone de Test ---
# Remplacez par le chemin d'une vraie image d'ordonnance ou de boîte dans votre dossier
image_test = "test_ordonnance.jpg" 

if os.path.exists(image_test) and df is not None:
    predictions = process_prescription_image(image_test, df)
    
    print("Résultats de l'analyse :")
    for pred in predictions:
        print(f"▶️ Trouvé : {pred['DENOMINATION']} (Code CIS: {pred['CIS']}) - Confiance : {pred['Confiance']:.2%}")
else:
    print(f"Pour tester, placez une image nommée '{image_test}' dans le dossier de votre notebook.")

Tokeniseur et labels sauvegardés avec succès.
